# QUEDA — Fiscalização de dados abertos do Crea-SP

Executa a coleta e a análise **a partir do Colab** (o tráfego sai por IPs do Google, não da sua máquina).

**Os dados ficam no Google Drive** (célula 1). O Colab desconecta e apaga o disco; o Drive faz a coleta sobreviver.

**O caminho curto: células 1 → 2 → 4.** A célula 4 mostra o que falta, baixa **só** isso e entrega o relatório — mesmo que a coleta não complete. Repita a 4 quantas vezes precisar: cada volta só busca o buraco que sobrou.

**Camada de IA (opcional)**: roda um modelo local via Ollama dentro do próprio Colab — sem chave de API e sem que nenhum dado do conselho saia da máquina. Pular a célula 3 não tira nenhum número do relatório.

> Aviso: todo achado é **indício**, não prova. Confirme na fonte antes de qualquer denúncia.

In [ ]:
# 1) Google Drive — é isto que faz a coleta sobreviver à queda de sessão
# O Colab desconecta e apaga o disco. O clone do repositório é descartável
# (refazer leva segundos), mas `data/raw` guarda HORAS de coleta com throttle
# e `data/cache` guarda o que o modelo local já leu. Os dois vão para o Drive.
from google.colab import drive
drive.mount('/content/drive')

import os
PASTA = '/content/drive/MyDrive/queda'          # mude se quiser outro lugar
os.environ['QUEDA_DATA_DIR']   = f'{PASTA}/data'
os.environ['QUEDA_OUTPUT_DIR'] = f'{PASTA}/output'
for p in ('data/raw', 'data/staged', 'data/cache', 'output'):
    os.makedirs(f'{PASTA}/{p}', exist_ok=True)

# O modelo do Ollama NÃO vai para o Drive: o Drive é um sistema de arquivos
# FUSE e não suporta as operações que o Ollama faz no diretório de modelos —
# o serviço simplesmente não sobe. Ele fica em disco local e é rebaixado a
# cada sessão (poucos minutos), o que é o preço de o resto funcionar.
os.environ['OLLAMA_MODELS'] = '/content/ollama'
os.makedirs('/content/ollama', exist_ok=True)

print('dados persistidos em', PASTA)
!du -sh {PASTA}/data 2>/dev/null || true


In [ ]:
# 2) Obter o código (efêmero: o Drive guarda os dados, não o código)
# O repositório é PRIVADO, então o clone pede um token. Gere um fine-grained
# token com `Contents: Read-only` em github.com/settings/personal-access-tokens
# e cole no prompt — ele NÃO fica salvo no notebook nem no Drive.
import os, subprocess
REPO = 'github.com/caiofvserra/queda.git'
BRANCH = 'claude/crea-sp-audit-tool-m9nw7t'
if os.path.exists('/content/queda'):
    !cd /content/queda && git fetch -q origin {BRANCH} && git reset -q --hard origin/{BRANCH}
else:
    r = subprocess.run(['git','clone','--depth','1','-b',BRANCH,f'https://{REPO}'],
                       cwd='/content')
    if r.returncode != 0:
        from getpass import getpass
        tok = getpass('Token GitHub (Contents: read-only): ')
        subprocess.run(['git','clone','--depth','1','-b',BRANCH,
                        f'https://{tok}@{REPO}'], cwd='/content', check=True)
        del tok
%cd /content/queda
!pip -q install -r requirements.txt
!pip -q install networkx   # intermediação exata no grafo da rede


In [ ]:
# 3) Camada de IA rodando LOCAL (Ollama)
# Nenhum número do relatório depende disto: todo cálculo é determinístico.
# O modelo entra só onde regra nenhuma serve — enquadrar o resíduo da
# taxonomia, ler a prosa das atas e redigir o resumo executivo.
import os
os.environ['QUEDA_LLM_MODELO'] = 'qwen2.5:7b-instruct'   # sem GPU: 'llama3.2:3b'

# Existe GPU nesta sessão? (Ambiente de execução -> Alterar tipo -> GPU)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "SEM GPU"

# pciutils/lshw ANTES do Ollama: sem `lspci` o instalador não enxerga a placa,
# pula as dependências de CUDA e o modelo roda em CPU — dez vezes mais lento,
# sem erro nenhum, só um WARNING no meio do log.
!sudo apt-get -qq update && sudo apt-get -qq install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh

# O servidor sobe aqui, no processo do próprio notebook — e NÃO dentro de um
# `!python`. Um processo que sobrevive ao comando e segura o pipe de saída faz
# a célula girar para sempre, mesmo com o trabalho concluído.
import subprocess, time, requests

def ollama_no_ar(t=3):
    try:
        return requests.get('http://localhost:11434/api/tags', timeout=t).ok
    except Exception:
        return False

if not ollama_no_ar():
    _log = open('/tmp/ollama-queda.log', 'wb')
    subprocess.Popen(['ollama', 'serve'], stdin=subprocess.DEVNULL,
                     stdout=_log, stderr=_log, start_new_session=True)
    for _ in range(60):
        time.sleep(1)
        if ollama_no_ar(1):
            break
print('Ollama no ar:', ollama_no_ar())

# Já com o servidor de pé, este comando só baixa o modelo e confere se ele
# carregou em GPU ou em CPU. Termina e devolve o controle.
!python main.py llm


In [ ]:
# 4) RODAR SÓ O QUE FALTA — inventário, coleta do que falta, análise, relatório
#
# É esta a única célula que você precisa repetir. Ela faz, nesta ordem:
#   1. `faltando`  — lê o disco e diz quais exercícios ainda não vieram. Sem rede.
#   2. `all`       — vai à rede SÓ por esses exercícios (checkpoint por janela em
#                    data/raw/checkpoints) e depois analisa e gera o relatório.
#
# Se o portal bloquear por ritmo no meio, a coleta para MAS A ANÁLISE ACONTECE:
# o relatório sai sobre o que já está em disco, e a frente `diagnostico` diz o
# que ficou faltando. Rode a célula de novo mais tarde — ela retoma pelo buraco.
#
# Diagnóstico da API (qual prefixo serve o quê, quais Creas respondem, o
# contrato OpenAPI) NÃO fica no caderno: são ~60 requisições num portal que
# bloqueia por rajada. Quando precisar, uma vez: `!python main.py sondar`.
#
# A trava abaixo existe porque o modo de falhar era silencioso: sem
# QUEDA_OUTPUT_DIR, config.py cai para <repo>/output — que o Colab apaga na
# desconexão. A rodada terminava, o relatório ia para o disco efêmero, e a
# célula de download zipava a pasta do Drive, vazia. Reconectou e pulou a célula 1?
# É exatamente esse o caso. Aqui isso vira erro, não surpresa no fim.
import os
if not os.environ.get('QUEDA_OUTPUT_DIR') or not os.environ.get('QUEDA_DATA_DIR'):
    raise SystemExit(
        'Rode a célula 1 antes desta. Sem QUEDA_DATA_DIR/QUEDA_OUTPUT_DIR a\n'
        'coleta grava no disco efêmero do Colab e se perde na desconexão.')
print('gravando em', os.environ['QUEDA_OUTPUT_DIR'])

# --desde auto: sonda anos retroativamente até a API esvaziar (meta: pré-2016)
!python main.py faltando --desde auto --ate 2026
# --comparar-creas liga a frente "Comparação entre Creas": ComparativoDespesa
# dos outros conselhos na mesma API (~28 hosts × 3 exercícios). Opt-in porque
# custa requisições; tire a flag se o portal estiver bloqueando por ritmo.
!python main.py all      --desde auto --ate 2026 --retomar --comparar-creas

In [ ]:
# 5) Ver o relatório aqui mesmo
# Cada rodada fica guardada em output/rodadas/<data_hora>/; a raiz tem sempre
# a mais recente. O relatório abre com "o que mudou desde a rodada anterior".
import os
from IPython.display import HTML, display
SAIDA = os.environ['QUEDA_OUTPUT_DIR']
display(HTML(open(os.path.join(SAIDA, 'report.html')).read()))


In [ ]:
# 6) Baixar o resultado — com conferência antes de zipar.
# Um zip vazio parece sucesso e só se descobre do outro lado. Esta célula
# checa se há relatório, e se não houver procura no lugar onde ele cai
# quando a célula 1 não foi rodada (<repo>/output) antes de desistir.
import os, pathlib

SAIDA = pathlib.Path(os.environ.get('QUEDA_OUTPUT_DIR', '/content/queda/output'))
FALLBACK = pathlib.Path('/content/queda/output')

def tem_resultado(d):
    return d.is_dir() and ((d / 'report.html').exists() or any(d.glob('rodadas/*/report.html')))

origem = None
if tem_resultado(SAIDA):
    origem = SAIDA
elif tem_resultado(FALLBACK):
    origem = FALLBACK
    print('ATENÇÃO: o relatório está em', FALLBACK, 'e NÃO no Drive.')
    print('A célula 1 não rodou nesta sessão. Copiando para o Drive antes de zipar.')
    !cp -r "{FALLBACK}/." "{SAIDA}/"
    origem = SAIDA

if origem is None:
    raise SystemExit(
        f'Nenhum relatório em {SAIDA} nem em {FALLBACK}.\n'
        'A rodada não chegou à etapa de relatório. Verifique o fim da saída da\n'
        'célula 4 e, se a coleta parou no meio, rode-a de novo com --retomar.')

rodadas = sorted(p.parent.name for p in origem.glob('rodadas/*/report.html'))
print(f'{len(rodadas)} rodada(s) guardada(s):', ', '.join(rodadas) or '(só a raiz)')

# Junto com o relatório vai a EVIDÊNCIA da coleta: o ledger de requisições e os
# corpos das respostas 4xx do portal (data/raw/http-4xx). É o que diz por que
# uma janela foi recusada — sem isso a análise de fora só vê "HTTP 400".
DADOS = pathlib.Path(os.environ.get('QUEDA_DATA_DIR', '/content/queda/data'))
extras = [p for p in (DADOS / 'raw' / 'http-4xx',
                      DADOS / 'raw' / 'swagger',
                      DADOS / 'raw' / 'sondar',
                      DADOS / 'raw' / 'tcu-acordaos') if p.exists()]

# Os CSVs staged PEQUENOS vão junto. Sem eles, quem analisa o resultado de fora
# não consegue conferir o FORMATO de uma coluna nova — foi assim que o
# cruzamento com o TCU ficou sem diagnóstico na rodada 13: os 763 responsáveis
# vieram, nenhum casou, e não havia como saber se a máscara do CPF estava
# alinhada. `pagamentos.csv` (255 MB) e `sancoes.csv` (29 MB) ficam de fora
# pelo tamanho; delas vai só o cabeçalho e uma amostra.
TETO_STAGED = 3_000_000
amostras = pathlib.Path('/content/amostras'); amostras.mkdir(exist_ok=True)
for csv in sorted((DADOS / 'staged').glob('*.csv')):
    if csv.stat().st_size <= TETO_STAGED:
        extras.append(csv)
    else:
        destino = amostras / f'{csv.stem}.amostra.csv'
        with open(csv, encoding='utf-8', errors='replace') as f, \
                open(destino, 'w', encoding='utf-8') as saida:
            for i, linha in enumerate(f):
                if i > 50:
                    break
                saida.write(linha)
        print(f'{csv.name}: {csv.stat().st_size/1e6:.0f} MB — só cabeçalho e 50 linhas')
!cd "{origem.parent}" && zip -qr /content/resultado_queda.zip "{origem.name}/"
for extra in extras:
    !cd "{DADOS.parent}" && zip -qr /content/resultado_queda.zip "{extra.relative_to(DADOS.parent)}"
if any(amostras.iterdir()):
    !cd /content && zip -qr /content/resultado_queda.zip amostras
print('evidência incluída:', ', '.join(str(e.relative_to(DADOS.parent)) for e in extras) or 'nenhuma')
tam = pathlib.Path('/content/resultado_queda.zip').stat().st_size
print(f'zip: {tam/1e6:.1f} MB')
if tam < 10_000:
    raise SystemExit('O zip saiu vazio — não baixe. Confira a saída da célula 4.')

from google.colab import files
files.download('/content/resultado_queda.zip')


## Quando a sessão cair (e ela vai cair)

Reconecte e rode as células de novo, de cima a baixo. A célula 4 vai à rede apenas pelo que ainda não veio — não recomeça a coleta.

- `data/raw/` guarda a resposta de cada requisição; nada é rebaixado.
- **A retomada é por exercício em 11 das 17 etapas** — as que percorrem uma série anual (empenhos, pagamentos, contratos, aditivos, licitações, convênios, comparativo de despesa, despesas por centro de custo, demonstrativo, atas, comparativo de receita + balanço). Cada exercício coletado vira um checkpoint em `data/raw/checkpoints/<etapa>/`, e a etapa só vai à rede pelas janelas que faltam: cair em 2019 custa 2019, não a etapa toda.
- As outras 6 (`processos_itens`, `indenizacoes_endpoint`, `plano_contas`, `quadro_pessoal`, `mandatarios`, `pncp_contratos`) não têm checkpoint anual próprio — a retomada delas é o cache HTTP, e o critério continua sendo "vazio recoleta, cheio pula". O inventário diz isso na cara, em vez de fingir que conferiu exercício por exercício.
- **Etapa com CSV cheio e exercício faltando é refeita**, não pulada. Uma etapa que trouxe 2016–2025 e falhou em 2013–2015 tem linhas de sobra em `data/staged` — a versão anterior a considerava pronta e o buraco não fechava em rodada nenhuma. Refazer é barato: as janelas que já estão em checkpoint saem do disco, sem rede.
- Etapa que veio **vazia** é tentada de novo de propósito: dataset zerado quase sempre é bloqueio por rajada (403), e 403 nunca é cacheado. Janela que falhou também não vira checkpoint — falha não pode virar permanente. Idem para janela cujo formato de parâmetro o endpoint recusou (400): ela conta como **não coletada**, não como exercício vazio.
- `python main.py faltando` (o primeiro comando da célula 4) responde, sem gastar uma requisição, quantos exercícios faltam em cada etapa e quais são. `python main.py sondar` (fora do caderno, sob demanda) mede qual prefixo da API serve o quê e quais Creas respondem.
- Passadas 12h, o exercício **em curso** volta a contar como pendente — é o TTL do checkpoint do ano aberto, que muda todo dia. Rodar duas vezes dentro das 12h é quase de graça; depois disso, a rodada atualiza o exercício aberto em vez de não fazer nada.
- O portal **recusa** algumas janelas com HTTP 400 — na rodada de 04/09 foram o exercício 2026 em onze endpoints e 2010–2020 no `DespesasCentroCusto`. Recusa não é "o órgão não publica": é pergunta não aceita, e pode ser do nosso formato de parâmetro. Ela é anotada por 12h, para a rodada seguinte não queimar ritmo insistindo, e reaparece depois — a coluna `recusados_pelo_portal` do inventário diz quais são.
- Para forçar recoleta de um período, `!python main.py limpar --etapa licitacoes,atas` apaga os marcadores de janela vazia dessas etapas (`--tudo` apaga também os checkpoints com dado). **Rodar sem `--retomar` não serve para isso**: `--retomar` só governa o pulo de etapa; a leitura do checkpoint por janela é incondicional. Para ignorar todos os checkpoints numa rodada, `QUEDA_CHECKPOINT=0`.

**Rodar de novo não apaga o relatório anterior.** Cada execução vira `output/rodadas/<data_hora>/`, e o relatório novo abre com o que mudou desde a última: achados que apareceram, que sumiram e valores que se moveram. O histórico completo está em `output/index.html`.

## Se uma célula parecer travada

Cheque antes de reiniciar — quase sempre o trabalho terminou:

```python
!tail -5 /tmp/ollama-queda.log          # o servidor subiu?
!curl -s localhost:11434/api/tags       # responde?
!ls -la "$QUEDA_DATA_DIR/staged" | tail # a coleta avançou?
```

Se o log e a API respondem, pode interromper a célula com segurança: o servidor do Ollama roda destacado e sobrevive à interrupção.

Só análise, sem tocar na rede: `!python main.py analyze` (aceita `--sem-llm`).

A aba **Diagnóstico da coleta** do relatório diz o que veio, o que faltou e qual análise ficou desligada por falta de qual fonte — comece por ela. Ela agora também diz **por que** cada fonte vazia veio vazia, lendo o status de cada requisição em `data/staged/_coleta_http.csv`: 403 do WAF e 404 de endpoint inexistente são limitação nossa; 200 com lista vazia é o órgão não publicando, e isso é achado de transparência.
